# 07 · Measuring TTFT, TPOT and goodput
Level L1 · Part 1 · core · ~20 min · builds on 06

This lesson uses a **stand-in engine** that runs on your laptop: `apps/07/engine.py`. It is not a language model. It sleeps for the times its settings say and streams one token per event. Its numbers are set in this notebook, so you can check that your measurement recovers them. The method is the point: the same client code works on a real server (lesson 08 starts a real one).

## Goal
By the end of this lesson you can measure TTFT, TPOT and inter-token latency from a streamed response, give p50 and p95 for a load run, and compute **goodput**: the requests per second that meet a latency SLO.

## Why it matters
- **Without it:** you time a whole answer with one clock and report one number. Two servers can finish in the same second and still feel completely different: one starts answering at once, the other sits silent and then streams fast. The next cell shows this.
- **With it:** the client records when each token arrives. That gives the wait before the first token (TTFT), the pace of the rest (TPOT and inter-token latency), and the share of requests that meet a target (goodput). These are the numbers users feel and the numbers an SLO is written in.
- **When you don't need it:** for a batch job where only the total time per job counts, one number is enough. For an interactive chat or a support desk, it is not.

### The wrong measure: one total time
Two made-up profiles, both with 60 output tokens and a total of about one second. The total cannot tell them apart. The arrival times can.

In [1]:
profiles = {"starts fast": {"ttft": 0.20, "tpot": 0.0136},
            "silent, then fast": {"ttft": 0.90, "tpot": 0.0017}}
for name, p in profiles.items():
    e2e = p["ttft"] + p["tpot"] * 59            # 59 gaps between 60 tokens
    print(f"{name:18} total {e2e:.2f} s   first token at {p['ttft']:.2f} s")

starts fast        total 1.00 s   first token at 0.20 s
silent, then fast  total 1.00 s   first token at 0.90 s


**What you should see:** both totals are about 1.0 s. Only the first token's time shows which answer the user waits for.

## The idea
Three clocks split a streamed answer. **TTFT** (time to first token) is the wait from sending the request to the first token; it covers queueing and prompt prefill. **ITL** (inter-token latency) is the gap between two consecutive tokens. **TPOT** (time per output token) is the average ITL after the first token: the decode pace. **Goodput** is a rate, not a time: the requests per second that meet every SLO target (for example, TTFT under 0.5 s and TPOT under 15 ms). A fast server that misses a target gets no credit for that request. Like a restaurant: TTFT is the wait for the first course, TPOT is the pace of the rest.

## Picture
```mermaid
flowchart LR
  S["Client sends request"] --> Q["Wait for a free slot (queue)"]
  Q --> P["Prefill: read the prompt"]
  P --> F["First token arrives: TTFT ends"]
  F --> D["Decode: one token per step (ITL, averaged as TPOT)"]
  D --> E["Last token: end-to-end latency"]
```

## Step by step
### Step 1: the example trace (made up)
The Brightside support desk sends a 1,500-token policy prompt plus a short question. The output length varies. The trace below is generated with a fixed seed. It is made-up data, not a real log.

In [2]:
import os, json, random, time, asyncio, subprocess, sys, socket
from pathlib import Path
import numpy as np
import httpx

N_REQUESTS = 30
rng = np.random.default_rng(7)
questions = np.clip(rng.lognormal(np.log(60), 0.5, N_REQUESTS).round(), 10, 300).astype(int)
outputs = np.clip(rng.lognormal(np.log(60), 0.6, N_REQUESTS).round(), 8, 250).astype(int)
trace = [{"prompt_tokens": 1500 + int(q), "output_tokens": int(o)} for q, o in zip(questions, outputs)]

Path("../data/07").mkdir(parents=True, exist_ok=True)
with open("../data/07/support_trace.jsonl", "w") as f:       # made-up trace, labelled as such
    for row in trace:
        f.write(json.dumps(row) + "\n")

print(f"{len(trace)} made-up requests; first three:")
for row in trace[:3]:
    print("  ", row)
print(f"median output {int(np.median(outputs))} tokens, median prompt {int(np.median([r['prompt_tokens'] for r in trace]))} tokens")

30 made-up requests; first three:
   {'prompt_tokens': 1560, 'output_tokens': 24}
   {'prompt_tokens': 1570, 'output_tokens': 45}
   {'prompt_tokens': 1552, 'output_tokens': 33}
median output 60 tokens, median prompt 1552 tokens


**What you should see:** 30 rows, each with a prompt of about 1,500 to 1,900 tokens and an output of a few dozen tokens. The file is written to `data/07/` and no other lesson reads it.

### Step 2: start the stand-in engine
The engine settings live in this cell and go to the server through a settings file, so there is one source for them. The engine reads 150 ms of prompt per 1,000 tokens, takes 6 ms per decode step plus 3 ms for each request it is running, and runs 4 requests at once. The rest wait in a queue. The server file `apps/07/engine.py` is written in the next cell. Port 8070 is this lesson's port.

In [3]:
ENGINE = {"prefill_ms_per_1k_tokens": 150, "decode_base_ms": 6,
          "decode_ms_per_running": 3, "max_running": 4}
SLO = {"ttft_s": 0.5, "tpot_s": 0.015}          # the targets a goodput count is checked against
PORT = 8070
BASE = f"http://127.0.0.1:{PORT}"
APPS = Path("../apps/07").resolve()
APPS.mkdir(parents=True, exist_ok=True)
(APPS / "settings.json").write_text(json.dumps(ENGINE))   # the engine reads its settings from this file

100

The server itself is a short FastAPI file. The notebook writes it to `apps/07/engine.py`, so the file is part of this lesson. It is a stand-in: it sleeps and streams, and it never runs a model.
`state["running"]` counts the requests holding one of the 4 slots right now. Each decode step reads that count when it starts, and costs a fixed base plus a little for every running request. So a busier batch slows every request down, which is how batching behaves in outline. Requests that wait for a slot are not counted here; they are queued.

In [4]:
%%writefile ../apps/07/engine.py
# A stand-in streaming engine for lesson 07. It is NOT a language model.
import asyncio, json
from pathlib import Path
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
ENGINE = json.loads((Path(__file__).parent / "settings.json").read_text())
app = FastAPI()
slots = asyncio.Semaphore(ENGINE["max_running"])
state = {"running": 0}
@app.get("/health")
async def health(): return {"ok": True}
@app.post("/generate")
async def generate(req: dict):
    async def stream():
        async with slots:                # waiting here is queueing: it shows up in TTFT
            state["running"] += 1
            try:
                await asyncio.sleep(req["prompt_tokens"] / 1000 * ENGINE["prefill_ms_per_1k_tokens"] / 1000)
                for i in range(req["output_tokens"]):
                    if i:                # one decode step: a fixed base plus a cost per running request
                        step_ms = ENGINE["decode_base_ms"] + ENGINE["decode_ms_per_running"] * state["running"]
                        await asyncio.sleep(step_ms / 1000)
                    yield f"data: {json.dumps({'token': i})}\n\n"
                yield "data: [DONE]\n\n"
            finally:
                state["running"] -= 1
    return StreamingResponse(stream(), media_type="text/event-stream")

Overwriting ../apps/07/engine.py


Start it from the notebook, check the port is free, and wait for the health check. If the process exits early, the wait stops at once and prints the log's location. The stop happens in the last cell, even after an error.

In [5]:
if socket.socket().connect_ex(("127.0.0.1", PORT)) == 0:
    raise RuntimeError(f"port {PORT} is busy: stop the other process first")
log = open("../data/07/engine.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "engine:app", "--app-dir", str(APPS),
                           "--port", str(PORT), "--log-level", "warning"],
                          stdout=log, stderr=subprocess.STDOUT)

def wait_healthy(timeout=20):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"engine exited with code {server.returncode}; see ../data/07/engine.log")
        try:
            if httpx.get(f"{BASE}/health", timeout=1).status_code == 200:
                return True
        except httpx.TransportError:
            time.sleep(0.1)
    raise TimeoutError(f"engine on port {PORT} did not answer in {timeout} s")

print("engine healthy:", wait_healthy())

engine healthy: True


### Step 3: one request, timed token by token
The client records the clock at each streamed token. TTFT is the first arrival. The gaps between arrivals are the ITLs. TPOT is their mean (the gap from the first token to the last, divided by the number of gaps). End-to-end latency is the last arrival.
Jupyter lets you use `await` and `async with` directly in a cell; in a script you would wrap them in `asyncio.run`.

In [6]:
async def one(client, req):
    t_send = time.perf_counter()
    arrivals = []
    async with client.stream("POST", f"{BASE}/generate", json=req) as r:
        async for line in r.aiter_lines():
            if line.startswith("data: ") and line != "data: [DONE]":
                arrivals.append(time.perf_counter() - t_send)
    n = len(arrivals)
    ttft, e2e = arrivals[0], arrivals[-1]
    tpot = (e2e - ttft) / (n - 1)                  # mean gap after the first token
    return {"n": n, "ttft": ttft, "tpot": tpot, "e2e": e2e, "end": time.perf_counter(),
            "itl": np.diff(arrivals)}              # every gap, for the tail

async with httpx.AsyncClient(timeout=120) as client:
    row = await one(client, trace[0])
print(f"{row['n']} tokens: TTFT {row['ttft']*1000:.0f} ms, TPOT {row['tpot']*1000:.1f} ms, "
      f"end-to-end {row['e2e']:.2f} s")
print(f"expected TTFT at no load: prompt {trace[0]['prompt_tokens']:,} tokens x "
      f"{ENGINE['prefill_ms_per_1k_tokens']} ms per 1k = {trace[0]['prompt_tokens'] * ENGINE['prefill_ms_per_1k_tokens'] / 1000:.0f} ms")

24 tokens: TTFT 284 ms, TPOT 9.3 ms, end-to-end 0.50 s
expected TTFT at no load: prompt 1,560 tokens x 150 ms per 1k = 234 ms


**What you should see:** the first request runs alone, so the engine's prefill time is the expected 234 ms, but the measured TTFT is higher. The gap is HTTP, scheduling and event-loop overhead. It is above the 234 ms prefill by roughly 50 to 135 ms in the test runs, and it changes from run to run. A real measurement contains that overhead too, so the client clock never shows the engine's prefill time exactly. TPOT is the 6 ms base step plus 3 ms for the request itself, about 9 ms, plus a little overhead.

### Step 4: a load run at three arrival rates
Requests arrive at random times (Poisson arrivals: random gaps with an average of 1/rate seconds, from a fixed seed), so bursts happen. At each rate, every request is timed as in step 3. The summary gives throughput (completed requests per second), goodput (requests per second that meet both SLO targets), and p50 and p95 of TTFT and TPOT. A percentile is the value that that share of requests stays under: p95 is the slow tail that users notice. The p95 of ITL uses every gap in the run, so one stall shows up there even when the mean looks fine. **In flight** is the average number of requests in the system, queued or running, over the run. The lesson uses it as a number to read, and lesson 26 (Little's law) explains where it comes from.

In [7]:
async def run_load(rate, seed=7):
    arrivals = random.Random(seed)
    async with httpx.AsyncClient(timeout=120) as client:
        tasks = []
        for req in trace:
            await asyncio.sleep(arrivals.expovariate(rate))    # gap to the next request
            tasks.append(asyncio.ensure_future(one(client, req)))
        return await asyncio.gather(*tasks)

def summarise(rate, rows, start):
    duration = max(r["end"] for r in rows) - start     # from the start of the run to the last request's end, arrival gaps included
    met = [r for r in rows if r["ttft"] <= SLO["ttft_s"] and r["tpot"] <= SLO["tpot_s"]]
    ttft = np.array([r["ttft"] for r in rows]); tpot = np.array([r["tpot"] for r in rows])
    itl = np.concatenate([r["itl"] for r in rows])
    return {"rate": rate, "throughput": len(rows) / duration, "goodput": len(met) / duration,
            "inflight": sum(r["e2e"] for r in rows) / duration,      # average requests in the system, queued or running (lesson 26)
            "met": len(met) / len(rows), "ttft50": np.percentile(ttft, 50),
            "ttft95": np.percentile(ttft, 95), "tpot50": np.percentile(tpot, 50),
            "tpot95": np.percentile(tpot, 95), "itl95": np.percentile(itl, 95)}

Now the three arrival rates. Each run takes a few seconds.

In [8]:
results = []
for rate in (1.5, 3.0, 4.5):
    start = time.perf_counter()
    rows = await run_load(rate)
    results.append(summarise(rate, rows, start))

print(f"{'req/s':>5} {'thru':>5} {'good':>5} {'met%':>5} {'in flight':>9}  {'TTFT p50':>8} {'p95':>6}  {'TPOT p50':>8} {'p95':>6}  {'ITL p95':>7}")
for s in results:
    print(f"{s['rate']:>5.1f} {s['throughput']:>5.2f} {s['goodput']:>5.2f} {s['met']:>5.0%} {s['inflight']:>9.1f}  "
          f"{s['ttft50']*1000:>6.0f}ms {s['ttft95']*1000:>5.0f}ms  "
          f"{s['tpot50']*1000:>6.1f}ms {s['tpot95']*1000:>5.1f}ms  {s['itl95']*1000:>5.1f}ms")

req/s  thru  good  met% in flight  TTFT p50    p95  TPOT p50    p95  ITL p95
  1.5  1.86  0.68   37%       2.6     237ms  1039ms    16.6ms  18.3ms   18.9ms
  3.0  2.51  0.08    3%       6.9    1544ms  2997ms    18.2ms  18.4ms   18.9ms
  4.5  2.54  0.00    0%       9.4    2705ms  5197ms    18.2ms  18.4ms   18.9ms


**What you should see:** throughput levels off near 2.4 req/s, because the engine is full. Goodput falls: about 0.7 req/s at 1.5 req/s offered, close to zero at 3.0, and zero at 4.5. TTFT grows as the queue builds (median from about 0.24 s to about 2.9 s). TPOT misses its 15 ms target even at the lowest rate: the p50 is about 17 ms, not the 9 ms of a lone request.

**Why TPOT misses the target:** each decode step costs 6 ms plus 3 ms for every request that holds a slot when the step starts. With three running, a step takes 15 ms, exactly the target. So any extra overhead, or a fourth request, pushes TPOT over the target.

**Reading the other columns:** "in flight" is the average number of requests in the system, queued or running. It can be larger than the 4 slots. Throughput can also read above the offered rate. The arrivals are random, and the total of 30 random gaps varies by about 18% from run to run (1 over the square root of 30), so a short run can arrive faster than its nominal rate. Longer runs settle nearer the offered rate. These numbers come from the stand-in's settings, not from a real engine. Real engines differ, but the same client code measures them (lesson 08 repeats this on a real server).

## At work
- **Cloudflare** reports tail latency, not averages: with prefill and decode on separate pools, its p90 inter-token latency fell from about 100 ms with high variance to 20 to 30 ms. Track p90 and p95. → lesson 34 (disaggregated prefill).
- **DistServe** (2024 paper) scores a system by the maximum request rate it serves within both TTFT and TPOT limits. That is goodput: a rate, not a speed. → lesson 27 (load tests against an SLO).
- **vLLM's `bench serve`** counts the requests that meet both targets with `--goodput ttft:500 tpot:15` (values in ms). Run it on your own deployment, not on a vendor number. → lesson 27.

Sources: Cloudflare, "High-performance LLMs" (https://blog.cloudflare.com/high-performance-llms/); DistServe, arXiv 2401.09670 (https://arxiv.org/abs/2401.09670); vLLM docs, `vllm bench serve` (https://docs.vllm.ai/en/latest/cli/bench/serve/).

## Common mistakes
**1. Timing a non-streamed answer and calling the total TTFT.** A request that waits for the full answer returns its first token only at the end. The next cell prints that number.

In [9]:
t0 = time.perf_counter()
async with httpx.AsyncClient(timeout=120) as client:
    r = await client.post(f"{BASE}/generate", json=trace[0])     # waits for the whole body
ttft_wrong = time.perf_counter() - t0
print(f"'TTFT' from a plain POST: {ttft_wrong:.2f} s, which is the end-to-end time")
print(f"streamed TTFT for the same request (step 3): {row['ttft']:.2f} s")

'TTFT' from a plain POST: 0.47 s, which is the end-to-end time
streamed TTFT for the same request (step 3): 0.28 s


**What you should see:** the "TTFT" from the plain POST is the whole response time. It is larger than the streamed TTFT printed on the line below it, which is the time to the first token of the same request. The plain POST waits for every token, so its total grows with the answer length, and for a long answer the gap is much larger. Stream the response to see the first token.

**2. Dividing by the gaps when the answer has one token.** TPOT divides by the number of gaps between tokens. One token has no gap, so the division fails. The trace keeps at least 8 output tokens, so `one()` is safe here.

In [10]:
ttft, e2e, n = 0.24, 0.24, 1                     # a one-token answer
tpot = (e2e - ttft) / (n - 1)                    # ZeroDivisionError: no gap between tokens

ZeroDivisionError: float division by zero

The fix: report TPOT only for answers with at least two tokens, and say so in the report. The next cell applies that fix to the same one-token answer.

In [11]:
ttft, e2e, n = 0.24, 0.24, 1                     # the same one-token answer
tpot = (e2e - ttft) / (n - 1) if n >= 2 else None
print("TPOT:", tpot, "(not reported: no gap between tokens)")

TPOT: None (not reported: no gap between tokens)


## Try it
Your SLO is only TTFT under 0.3 s; ignore TPOT. Note that 0.3 s is barely above the no-load prefill of these prompts (about 0.23 to 0.29 s), so very few requests can meet it once any queue forms. The engine from step 2 must still be running. Run one load at 3.0 req/s, count the requests that meet the TTFT target, and print goodput in requests per second. Use `run_load` and `trace` as in step 4. Hint: divide the count by the same duration that `summarise` uses, from the start of the run to the last request's end.

In [12]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [13]:
start = time.perf_counter()
rows = await run_load(3.0)
ok = [r for r in rows if r["ttft"] <= 0.3]
duration = max(r["end"] for r in rows) - start
print(f"TTFT-only goodput at 3.0 req/s: {len(ok) / duration:.2f} req/s ({len(ok)} of {len(rows)} requests)")

TTFT-only goodput at 3.0 req/s: 0.42 req/s (5 of 30 requests)


## Recap
- **TTFT** is the wait for the first token: queueing plus prefill. **ITL** is the gap between two tokens; **TPOT** is their mean after the first token.
- Measure at the client, from the request send to each streamed token. A plain total time hides both.
- **Goodput** counts only the requests that meet every SLO target, per second. Throughput counts all completions, so it can rise while goodput falls.
- Report p95 next to p50. The slow tail is what users remember.

**Interview:** "How do you measure TTFT and TPOT for a streaming LLM API, and why is goodput a better number than throughput?" Strong answer: time from the request send to the first streamed token, then the mean gap between later tokens; goodput counts only requests that meet both targets, so a fast system that misses its SLO gets no credit.

## Clean up
Stop the stand-in engine. This cell is the last one, so the server runs through the Try-it answer.

In [14]:
server.terminate()
server.wait(timeout=10)
log.close()
print("engine stopped; exit code", server.returncode)

engine stopped; exit code -15
